In [1]:
# Check whether easydiffraction is installed; install it if needed.
# Required for remote environments such as Google Colab.
import importlib.util

if importlib.util.find_spec('easydiffraction') is None:
    %pip install easydiffraction==2026.11.14 --extra-index-url https://download.easydiffraction.org/pypi/simple/


# PbSO₄ on D1A and a laboratory diffractometer

&nbsp; &nbsp; &nbsp; &nbsp; &nbsp; &nbsp; &nbsp; 

Fit PbSO₄ to two patterns at once, the Rietveld round robin's neutron pattern from D1A at ILL
and its laboratory X-ray pattern: one structure, two experiments.

It builds the catalogue case `pd-neut-xray-cwl_pbso4-d1a-lab` step by step: the same project the
command line fits and the app's Examples open.

## Import the library

In [2]:
import easydiffraction as edi

## Define the project

A new project is empty: it holds no structure and no experiment until you add them. The joint
fitting mode fits every experiment at once, against one set of structures.

In [3]:
project = edi.Project(name='pd-neut-xray-cwl_pbso4-d1a-lab')
project.analysis.fitting_mode = 'joint'

## Define the structure

Load the structure from EasyDiffraction's collection. Its CIF gives the space group, the cell and
the atom sites.

In [4]:
path = edi.download_data('pd-neut-xray-cwl_pbso4-d1a-lab-pbso4', destination='data')
project.structures.add_from_cif_path(path)
structure = project.structures['pbso4']

Data 'pd-neut-xray-cwl_pbso4-d1a-lab-pbso4' is at data/pd-neut-xray-cwl_pbso4-d1a-lab-pbso4.edi


In [5]:
project.display.structure('pbso4')

View(html=<156,505 characters>, height=660)

## Define the experiments

Create the two experiments from their measured data in EasyDiffraction's collection, then
describe each one's instrument, peak profile, background and the structures it links.

In [6]:
data_path = edi.download_data('pd-neut-xray-cwl_pbso4-d1a-lab-d1a-data', destination='data')
project.experiments.add_from_data_path(
    name='d1a',
    data_path=data_path,
    sample_form='powder',
    beam_mode='constant wavelength',
    radiation_probe='neutron',
    scattering_type='bragg',
)
d1a = project.experiments['d1a']
d1a.neutron_scattering_length = 'sears1992'
d1a.peak.type = 'cwl-pseudo-voigt-berar-baldinozzi'
d1a.instrument.name = 'D1A'
d1a.instrument.facility = 'ILL'
d1a.peak.broad_gauss_u = 0.2
d1a.peak.broad_gauss_v = -0.5
d1a.peak.broad_gauss_w = 0.4
d1a.peak.mixing_eta_0 = 0.5
d1a.peak.cutoff_fwhm = 30
d1a.instrument.setup_wavelength = 1.91
d1a.linked_structures.create(structure_id='pbso4', scale=1.5)
d1a.background.create(position=11, intensity=200.0)
d1a.background.create(position=15, intensity=200.0)
d1a.background.create(position=20, intensity=200.0)
d1a.background.create(position=30, intensity=200.0)
d1a.background.create(position=50, intensity=200.0)
d1a.background.create(position=70, intensity=200.0)
d1a.background.create(position=120, intensity=200.0)
d1a.background.create(position=153, intensity=200.0)

Data 'pd-neut-xray-cwl_pbso4-d1a-lab-d1a-data' is at data/pd-neut-xray-cwl_pbso4-d1a-lab-d1a.xye


In [7]:
data_path = edi.download_data('pd-neut-xray-cwl_pbso4-d1a-lab-lab-data', destination='data')
project.experiments.add_from_data_path(
    name='lab',
    data_path=data_path,
    sample_form='powder',
    beam_mode='constant wavelength',
    radiation_probe='xray',
    scattering_type='bragg',
)
lab = project.experiments['lab']
lab.xray_form_factor = 'it1992'
lab.xray_dispersion = 'sasaki1989'
lab.peak.type = 'cwl-pseudo-voigt-berar-baldinozzi'
lab.background.type = 'chebyshev'
lab.peak.broad_gauss_u = 0.02
lab.peak.broad_gauss_v = -0.02
lab.peak.broad_gauss_w = 0.008
lab.peak.mixing_eta_0 = 0.5
lab.peak.asym_beba_limit = 160
lab.peak.cutoff_fwhm = 48
lab.instrument.setup_wavelength = 1.54056
lab.instrument.setup_wavelength_2 = 1.5444
lab.instrument.setup_wavelength_2_to_1_ratio = 0.5
lab.instrument.setup_polarization_coefficient = 0.5
lab.instrument.setup_monochromator_twotheta = 26.5650511771
lab.background_x_min = 15
lab.background_x_max = 160
lab.excluded_regions.create(start=0, end=15)
lab.excluded_regions.create(start=160, end=180)
lab.linked_structures.create(structure_id='pbso4', scale=0.001)
terms = []
for coef in (140.0, 0, 0, 0, 0, 0):
    term = edi.PolynomialTerm()
    term.order = len(terms)
    term.coef = coef
    terms.append(term)
lab.background_terms = terms

Data 'pd-neut-xray-cwl_pbso4-d1a-lab-lab-data' is at data/pd-neut-xray-cwl_pbso4-d1a-lab-lab.xye


In [8]:
project.display.pattern('d1a')

View(html=<383,524 characters>, height=480)

## Perform the analysis

Both patterns fit together. diffraction-lib's tutorial of this case frees everything at once,
and also the D1A wavelength, which this case keeps fixed.

### Scales, 2θ zeros and cell

In [9]:
d1a.linked_structures['pbso4'].scale.free = True
lab.linked_structures['pbso4'].scale.free = True
d1a.instrument.calib_twotheta_offset.free = True
lab.instrument.calib_twotheta_offset.free = True
structure.cell.length_a.free = True
structure.cell.length_b.free = True
structure.cell.length_c.free = True

In [10]:
project.display.parameters.free()

#,datablock,category,entry,parameter,value,s.u.,min,max,units
1,pbso4,cell,,length_a,8.47,0,0,99,Å
2,pbso4,cell,,length_b,5.39,0,0,99,Å
3,pbso4,cell,,length_c,6.95,0,0,99,Å
4,d1a,instrument,,calib_twotheta_offset,0,0,-inf,inf,deg
5,d1a,linked_structure,pbso4,scale,1.5,0,-inf,inf,
6,lab,instrument,,calib_twotheta_offset,0,0,-inf,inf,deg
7,lab,linked_structure,pbso4,scale,0.001,0,-inf,inf,


In [11]:
project.analysis.fit()
project.display.fit.results()

edi fit: joint project pd-neut-xray-cwl_pbso4-d1a-lab, 2 banks (d1a, lab), 8709 pts fitted / 8911 loaded, 7 free
   iter   time (s)   Rwp (%)  reduced χ²   change
                       64.60       483.00
      1      0.202     27.86        89.85   81.4% ↓
      2      0.227     27.34        86.55   3.7% ↓
      3      0.252     26.85        83.45   3.6% ↓
      4      0.277     26.38        80.57   3.4% ↓
      5      0.302     25.95        77.92   3.3% ↓
      6      0.326     25.50        75.27   3.4% ↓
      7      0.351     25.02        72.45   3.7% ↓
      8      0.375     24.52        69.58   4.0% ↓
      9      0.400     24.01        66.72   4.1% ↓
     10      0.425     23.47        63.76   4.4% ↓
     11      0.449     22.90        60.71   4.8% ↓
     12      0.474     22.34        57.76   4.9% ↓
     13      0.498     21.80        55.02   4.7% ↓
     14      0.523     21.31        52.56   4.5% ↓
     15      0.547     20.89        50.52   3.9% ↓
     16      0.572     20.57 

View(html=<94,448 characters>, height=376)

In [12]:
project.display.pattern('d1a')

View(html=<383,804 characters>, height=480)

In [13]:
project.display.pattern('lab')

View(html=<740,181 characters>, height=480)

### Backgrounds

D1A's points and the laboratory pattern's Chebyshev terms.

In [14]:
for point in d1a.background:
    point.intensity.free = True
for term in lab.background_terms:
    term.coef.free = True

In [15]:
project.analysis.fit()
project.display.fit.results()

edi fit: joint project pd-neut-xray-cwl_pbso4-d1a-lab, 2 banks (d1a, lab), 8709 pts fitted / 8911 loaded, 21 free
   iter   time (s)   Rwp (%)  reduced χ²   change
                       20.03        46.51
      1      0.235     19.11        42.33   9.0% ↓
      2      0.280     19.11        42.33   
  done · converged · 2 iters · reduced χ² 42.33 · Rwp 19.11 % · 0.31 s


View(html=<110,499 characters>, height=376)

In [16]:
project.display.pattern('d1a')

View(html=<409,872 characters>, height=480)

In [17]:
project.display.pattern('lab')

View(html=<792,319 characters>, height=480)

### Peak widths

In [18]:
for parameter in (
    d1a.peak.broad_gauss_u,
    d1a.peak.broad_gauss_v,
    d1a.peak.broad_gauss_w,
    d1a.peak.mixing_eta_0,
    lab.peak.broad_gauss_u,
    lab.peak.broad_gauss_v,
    lab.peak.broad_gauss_w,
    lab.peak.mixing_eta_0,
):
    parameter.free = True

In [19]:
project.analysis.fit()
project.display.fit.results()

edi fit: joint project pd-neut-xray-cwl_pbso4-d1a-lab, 2 banks (d1a, lab), 8709 pts fitted / 8911 loaded, 29 free
   iter   time (s)   Rwp (%)  reduced χ²   change
                       19.11        42.37
      1      0.328     11.62        15.68   63.0% ↓
      2      0.396     11.49        15.31   2.3% ↓
      3      0.467     11.42        15.14   1.1% ↓
      4      0.544     11.40        15.08   0.4% ↓
      5      0.625     11.39        15.06   0.1% ↓
      6      0.707     11.39        15.05   
      7      0.804     11.39        15.05   
      8      0.901     11.39        15.05   
      9      1.022     11.39        15.05   
  done · converged · 9 iters · reduced χ² 15.05 · Rwp 11.39 % · 1.12 s


View(html=<124,589 characters>, height=376)

In [20]:
project.display.pattern('d1a')

View(html=<409,922 characters>, height=480)

In [21]:
project.display.pattern('lab')

View(html=<792,582 characters>, height=480)

### Peak asymmetry, D1A

The D1A pattern's four Bérar-Baldinozzi terms.

In [22]:
for parameter in (
    d1a.peak.asym_beba_a0,
    d1a.peak.asym_beba_b0,
    d1a.peak.asym_beba_a1,
    d1a.peak.asym_beba_b1,
):
    parameter.free = True

In [23]:
project.analysis.fit()
project.display.fit.results()

edi fit: joint project pd-neut-xray-cwl_pbso4-d1a-lab, 2 banks (d1a, lab), 8709 pts fitted / 8911 loaded, 33 free
   iter   time (s)   Rwp (%)  reduced χ²   change
                       11.39        15.06
      1      0.570      9.65        10.81   28.2% ↓
      2      0.619      9.64        10.79   0.2% ↓
      3      0.667      9.64        10.79   
      4      0.722      9.64        10.79   
      5      0.776      9.64        10.79   
      6      0.846      9.64        10.79   
  done · converged · 6 iters · reduced χ² 10.79 · Rwp 9.64 % · 0.90 s


View(html=<128,248 characters>, height=376)

In [24]:
project.display.pattern('d1a')

View(html=<409,994 characters>, height=480)

In [25]:
project.display.pattern('lab')

View(html=<792,782 characters>, height=480)

### Peak asymmetry, laboratory

The laboratory pattern's first-order terms.

In [26]:
lab.peak.asym_beba_a0.free = True
lab.peak.asym_beba_b0.free = True

In [27]:
project.analysis.fit()
project.display.fit.results()

edi fit: joint project pd-neut-xray-cwl_pbso4-d1a-lab, 2 banks (d1a, lab), 8709 pts fitted / 8911 loaded, 35 free
   iter   time (s)   Rwp (%)  reduced χ²   change
                        9.64        10.79
      1      0.467      9.45        10.37   3.9% ↓
  done · converged · 1 iters · reduced χ² 10.37 · Rwp 9.45 % · 0.53 s


View(html=<129,977 characters>, height=376)

In [28]:
project.display.pattern('d1a')

View(html=<410,163 characters>, height=480)

In [29]:
project.display.pattern('lab')

View(html=<792,734 characters>, height=480)

### Atom positions

The neutron and X-ray patterns together see the oxygen as well as the lead, so all free
coordinates fit at once.

In [30]:
for site in structure.atom_sites:
    for coordinate in (site.fract_x, site.fract_y, site.fract_z):
        coordinate.free = not coordinate.symmetry_constrained

In [31]:
project.analysis.fit()
project.display.fit.results()

edi fit: joint project pd-neut-xray-cwl_pbso4-d1a-lab, 2 banks (d1a, lab), 8709 pts fitted / 8911 loaded, 46 free
   iter   time (s)   Rwp (%)  reduced χ²   change
                        9.45        10.39
      1      0.762      8.47         8.34   19.7% ↓
  done · converged · 1 iters · reduced χ² 8.34 · Rwp 8.47 % · 0.87 s


View(html=<146,737 characters>, height=376)

In [32]:
project.display.pattern('d1a')

View(html=<410,080 characters>, height=480)

In [33]:
project.display.pattern('lab')

View(html=<792,755 characters>, height=480)

### Displacements

In [34]:
for site in structure.atom_sites:
    site.adp_iso.free = True

In [35]:
project.analysis.fit()
project.display.fit.results()

edi fit: joint project pd-neut-xray-cwl_pbso4-d1a-lab, 2 banks (d1a, lab), 8709 pts fitted / 8911 loaded, 51 free
   iter   time (s)   Rwp (%)  reduced χ²   change
                        8.47         8.34
      1      0.405      5.80         3.92   53.1% ↓
      2      0.469      5.80         3.92   
  done · converged · 2 iters · reduced χ² 3.92 · Rwp 5.80 % · 0.52 s


View(html=<153,241 characters>, height=376)

In [36]:
project.display.pattern('d1a')

View(html=<410,450 characters>, height=480)

In [37]:
project.display.pattern('lab')

View(html=<792,811 characters>, height=480)

In [38]:
project.display.fit.correlations()

,1,2,3,4,5
1 d1a.peak.broad_gauss_u,,,,,
2 d1a.peak.broad_gauss_v,-0.93,,,,
3 d1a.peak.broad_gauss_w,0.79,-0.95,,,
4 d1a.peak.asym_beba_a1,0.28,-0.28,0.27,,
5 d1a.peak.asym_beba_b1,0.32,-0.25,0.19,0.94,


In [39]:
project.display.structure('pbso4')

View(html=<167,595 characters>, height=660)

## Report

The HTML report draws the measured, calculated and difference patterns with the reflection
positions, and lists the structures, the experiments and the fit. Open it in a browser.

In [40]:
project.report.save_html('pd-neut-xray-cwl_pbso4-d1a-lab.html')

Saving report to 'pd-neut-xray-cwl_pbso4-d1a-lab.html'


## Save the project

In [41]:
project.save_as('pd-neut-xray-cwl_pbso4-d1a-lab')

Saving project to 'pd-neut-xray-cwl_pbso4-d1a-lab'
